# 02 Predictive Modeling

Risk-scoring model for internal manufacturing failure prediction. The notebook emphasizes operating thresholds, recall, and false-negative risk rather than default classifier accuracy.

In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()))

import numpy as np
import pandas as pd
from src.config import SAMPLE_DIR
from src.preprocessing import (
    sample_training_data,
    feature_engineer,
    split_features_target,
    prepare_model_matrix,
)
from src.evaluate import threshold_table, select_threshold, risk_band

In [ ]:
try:
    df = pd.read_csv(SAMPLE_DIR / "bosch_training_sample.csv")
except FileNotFoundError:
    df = sample_training_data(n_rows=30000)

df = feature_engineer(df)
X, y = split_features_target(df)
X_model = prepare_model_matrix(X)
y.mean(), X_model.shape

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_model, y, test_size=0.25, random_state=42, stratify=y
)

baseline = DummyClassifier(strategy="prior", random_state=42)
baseline.fit(X_train, y_train)
baseline_scores = baseline.predict_proba(X_valid)[:, 1]

baseline_metrics = {
    "roc_auc": roc_auc_score(y_valid, baseline_scores),
    "pr_auc": average_precision_score(y_valid, baseline_scores),
}
baseline_metrics

In [ ]:
try:
    from lightgbm import LGBMClassifier

    scale_pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
    model = LGBMClassifier(
        n_estimators=250,
        learning_rate=0.04,
        num_leaves=31,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
    )
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier

    model = HistGradientBoostingClassifier(
        learning_rate=0.06, max_iter=160, l2_regularization=0.05, random_state=42
    )

model.fit(X_train, y_train)
scores = (
    model.predict_proba(X_valid)[:, 1]
    if hasattr(model, "predict_proba")
    else model.decision_function(X_valid)
)
if scores.min() < 0 or scores.max() > 1:
    scores = 1 / (1 + np.exp(-scores))

In [ ]:
thresholds = threshold_table(y_valid, scores, thresholds=[0.03, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50])
thresholds

In [ ]:
selected_threshold = select_threshold(thresholds, min_recall=0.70, max_inspection_rate=0.40)
pred = (scores >= selected_threshold).astype(int)
metrics = {
    "selected_threshold": selected_threshold,
    "precision": precision_score(y_valid, pred, zero_division=0),
    "recall": recall_score(y_valid, pred, zero_division=0),
    "f1": f1_score(y_valid, pred, zero_division=0),
    "roc_auc": roc_auc_score(y_valid, scores),
    "pr_auc": average_precision_score(y_valid, scores),
    "confusion_matrix": confusion_matrix(y_valid, pred).tolist(),
}
metrics

In [ ]:
scored = pd.DataFrame({"risk_score": scores, "actual_response": y_valid.values})
scored["risk_band"] = scored["risk_score"].map(risk_band)
scored["risk_band"].value_counts().to_frame("components")

## Cost-Sensitive Interpretation

False negatives are higher-risk than false positives because missed failures may continue through production or reach customers. False positives still matter because they consume inspection capacity and may create flow disruption. Threshold policy should therefore be approved jointly by quality and manufacturing leaders.